In [0]:
# %pip install -U \
#     langchain>=0.3.0 \
#     langchain-core>=0.3.0 \
#     databricks-langchain>=0.3.0
# %pip install databricks-vectorsearch
# dbutils.library.restartPython()


# %pip install \
#     "langchain>=0.3.0,<1.0.0" \
#     "langchain-core>=0.3.0,<1.0.0" \
#     "langchain-community>=0.3.0,<1.0.0" \
#     "databricks-langchain>=0.3.0,<1.0.0" \
    
# %pip install databricks-vectorsearch
# dbutils.library.restartPython()

In [0]:
import os
from langchain_core.messages import HumanMessage
from langchain_community.chat_models import ChatDatabricks

from langchain_core.tools import tool
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain import DatabricksVectorSearch
import requests, json

# below ones for sending emails
import smtplib
import re
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

In [0]:
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    max_tokens=512,
    temperature=0.1
)

CLASSIFIER_PROMPT = """You are a workplace HR sensitivity classifier.

Analyze the employee's message and determine if it involves any of the following 
sensitive or organizationally detrimental situations:
- Harassment, bullying, intimidation, or hostile work environment
- Discrimination based on race, gender, religion, age, disability, or any protected class
- Sexual misconduct or inappropriate workplace behavior
- Legal threats, lawsuits, or regulatory complaints against the company
- Fraud, bribery, financial misconduct, or corruption
- Data breaches, confidential leaks, or intellectual property theft
- Whistleblowing or reporting of illegal organizational activity
- Retaliation against an employee for reporting misconduct
- Wrongful dismissal, unfair termination, or constructive dismissal
- Threats of violence or physical safety concerns
- Mental health crisis or employee distress signals
- Any situation exposing the organization to legal or reputational risk

Respond ONLY in this exact JSON format with no explanation:
{{
    "is_sensitive": true or false,
    "reason": "one sentence explaining why if sensitive, else null",
    "urgency": "critical, urgent, or normal"
}}

Employee message: {question}
"""


# ── Load credentials from Databricks Secrets ──────────────────────────────────
# GMAIL_SENDER    = dbutils.secrets.get("hr-agent-secrets", "gmail-sender")
# GMAIL_PASSWORD  = dbutils.secrets.get("hr-agent-secrets", "gmail-app-password")
# GMAIL_RECIPIENT = dbutils.secrets.get("hr-agent-secrets", "gmail-recipient")

## BUILD THE AGENT WITH TOOLS

#### * DEFINE AGENT TOOLS

In [0]:
vector_store = DatabricksVectorSearch(
    endpoint="hr_policy_vs_endpoint",
    index_name="workspace.hr.policy_chunks_index",
)

def classify_sensitivity(question: str) -> dict:
    response = llm.invoke([
        HumanMessage(content=CLASSIFIER_PROMPT.format(question=question))
    ])
    
    import json, re
    try:
        # Extract JSON from response
        json_match = re.search(r'\{.*\}', response.content, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except (json.JSONDecodeError, AttributeError):
        pass
    
    # Safe default if parsing fails
    return {"is_sensitive": False, "reason": None, "urgency": "normal"}


# --- Tool 1: HR Policy Retrieval ---
@tool
def search_hr_policy(query: str) -> str:
    """
    Search HR policy documents to answer employee questions about 
    leave, conduct, benefits, disciplinary procedures, and other HR topics.
    Always use this tool first before escalating.
    """
    results = vector_store.similarity_search(query, k=4)
    if not results:
        return "No relevant HR policy found for your query."
    
    formatted = []
    for doc in results:
        source = doc.metadata.get("filename", "Unknown")
        section = doc.metadata.get("section_heading", "")
        formatted.append(f"[{source} — {section}]:\n{doc.page_content}")
    
    return "\n\n".join(formatted)


# --- Tool 2: Escalate to Human HR ---
@tool
def escalate_to_hr_human(employee_email: str, issue_summary: str, urgency: str = "normal") -> str:
    """
    Escalates the employee's issue to a human HR representative.
    Use this when:
    - The employee explicitly asks to speak with a human
    - The issue involves harassment, discrimination, or legal matters
    - Policy is unclear or contradictory after searching
    - The question involves termination, grievances, or disciplinary actions
    - The employee appears distressed or frustrated

    Args:
        employee_email: The employee's work email address
        issue_summary: Clear summary of the issue to hand off to HR
        urgency: 'normal', 'urgent', or 'critical'
    """
    # Option A: Create a ticket via ServiceNow/Jira API
    ticket_payload = {
        "caller_id": employee_email,
        "short_description": f"[HR Bot Escalation] {issue_summary}",
        "priority": "1" if urgency == "critical" else "2",
        "assignment_group": "HR_Support_Team",
        "description": issue_summary
    }
    
    # POST to your ticketing system
    # response = requests.post(SERVICENOW_URL, json=ticket_payload, auth=(...))
    
    # Option B: Send email notification via SendGrid/SMTP
    # send_email(to="hr-support@company.com", body=issue_summary)
    
    # Option C: Post to Slack HR channel
    slack_message = {
        "text": f":sos: *HR Escalation Needed*\n*Employee:* {employee_email}\n*Issue:* {issue_summary}\n*Urgency:* {urgency}"
    }
    # requests.post(SLACK_WEBHOOK_URL, json=slack_message)
    
    ticket_id = f"HR-{hash(employee_email) % 99999}"  # Replace with real ticket ID

    urgency_sla = {
        "critical": "2 hours",
        "urgent":   "4 hours",
        "normal":   "1 business day"
    }.get(urgency, "1 business day")

    return (
        f"I've raised a ticket (ID: **{ticket_id}**) with the HR team on your behalf. "
        f"An HR representative will reach out to {employee_email} within {urgency_sla}. "
        f"Please reference ticket {ticket_id} in any follow-up communication."
    )


@tool
def send_sensitive_alert_email(
    employee_email: str,
    question_asked: str,
    sensitivity_reason: str,
    urgency: str = "normal"
) -> str:
    """
    Send an urgent email alert to HR admin when the employee's message has been 
    classified by AI as sensitive or potentially detrimental to the organization.
    The AI has already determined this message requires immediate HR attention.
    Use the sensitivity_reason provided by the classifier as the reason for flagging.

    Args:
        employee_email: Email of the employee who raised the issue
        question_asked: The original message or question from the employee
        sensitivity_reason: The AI-determined reason this was flagged as sensitive
        urgency: 'normal', 'urgent', or 'critical' as determined by the classifier
    """

    GMAIL_SENDER    = os.environ["GMAIL_SENDER"]
    GMAIL_PASSWORD  = os.environ["GMAIL_PASSWORD"]
    GMAIL_RECIPIENT = os.environ["GMAIL_RECIPIENT"]
    
    urgency_emoji = {
        "critical": "🔴",
        "urgent":   "🟡",
        "normal":   "🟢"
    }.get(urgency,  "🟢")

    subject = f"{urgency_emoji} [{urgency.upper()}] AI-Flagged HR Sensitive Issue — {employee_email}"

    body = f"""
    <html>
    <body style="font-family: Arial, sans-serif; padding: 20px;">

        <h2 style="color: #c0392b;">⚠️ AI-Flagged Sensitive HR Issue</h2>
        <p style="color: #7f8c8d;">
            This alert was automatically generated because the HR assistant AI 
            classified the following employee message as sensitive.
        </p>

        <table style="border-collapse: collapse; width: 100%;">
            <tr>
                <td style="padding: 8px; font-weight: bold; width: 220px;">Employee Email</td>
                <td style="padding: 8px;">{employee_email}</td>
            </tr>
            <tr style="background-color: #f2f2f2;">
                <td style="padding: 8px; font-weight: bold;">Urgency</td>
                <td style="padding: 8px;">{urgency_emoji} {urgency.upper()}</td>
            </tr>
            <tr>
                <td style="padding: 8px; font-weight: bold;">Employee Message</td>
                <td style="padding: 8px;">{question_asked}</td>
            </tr>
            <tr style="background-color: #f2f2f2;">
                <td style="padding: 8px; font-weight: bold;">AI Flagging Reason</td>
                <td style="padding: 8px;">{sensitivity_reason}</td>
            </tr>
            <tr>
                <td style="padding: 8px; font-weight: bold;">Flagged At</td>
                <td style="padding: 8px;">
                    {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S UTC')}
                </td>
            </tr>
        </table>

        <br/>
        <p style="color: #7f8c8d; font-size: 12px;">
            This alert was generated by the MS-Square HR Policy Assistant AI. 
            Please review and take appropriate action promptly.
        </p>
    </body>
    </html>
    """

    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = subject
        msg["From"]    = GMAIL_SENDER
        msg["To"]      = GMAIL_RECIPIENT
        msg.attach(MIMEText(body, "html"))

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(GMAIL_SENDER, GMAIL_PASSWORD)
            server.sendmail(GMAIL_SENDER, GMAIL_RECIPIENT, msg.as_string())

        return (
            f"I've flagged your message as sensitive and notified the HR team immediately. "
            f"Someone will reach out to {employee_email} within "
            f"{'2 hours' if urgency == 'critical' else '4 hours' if urgency == 'urgent' else '1 business day'}. "
            f"You are not alone — HR will support you through this."
        )
    except Exception as e:
        return f"Could not send alert automatically. Please contact HR directly at hr@mssquare.com. Error: {str(e)}"

# # --- Tool 3: Check Employee Leave Balance (example action) ---
# @tool  
# def get_employee_leave_balance(employee_id: str) -> str:
#     """Fetch the current leave balance for an employee from HR systems."""
#     # Query your HR system (Workday, SAP, etc.)
#     # result = requests.get(f"{HR_API_URL}/employees/{employee_id}/leave")
#     return f"Annual Leave: 12 days remaining | Sick Leave: 8 days remaining"

#### * ASSEMBLE THE AGENT

In [0]:
import mlflow
from langchain_core.tools import tool                        # ✅ tool decorator
from langchain_core.prompts import ChatPromptTemplate        # ✅ prompt
from langchain_core.messages import AIMessage, HumanMessage  # ✅ chat history
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor


tools = [search_hr_policy, escalate_to_hr_human, send_sensitive_alert_email]

SYSTEM_PROMPT = """You are a helpful and professional HR Policy Assistant for MS-Square Limited LLC.

Your responsibilities:
- Answer employee questions about HR policies accurately using the search_hr_policy tool
- Always cite the source policy document and section in your answer
- Be empathetic, concise, and professional at all times

You have access to these tools:
- search_hr_policy: search HR policy documents for policy questions
- escalate_to_hr_human: raise a support ticket for HR team
- send_sensitive_alert_email: send an immediate email alert for sensitive issues
- ask_hr_genie: query live HR data like leave balances

Escalation rules — use escalate_to_hr_human immediately when:
- The employee explicitly asks to speak with a human or HR representative
- The issue involves harassment, discrimination, retaliation, or legal matters
- You cannot find a clear answer after searching the policy documents
- The question involves termination, formal grievances, or disciplinary action
- The employee appears distressed or the situation is time-sensitive

CRITICAL — Always use send_sensitive_alert_email immediately when the employee mentions:
- Harassment, bullying, or discrimination of any kind
- Legal threats, lawsuits, or regulatory complaints  
- Fraud, bribery, corruption, or financial misconduct
- Data breaches or unauthorized disclosure of confidential information
- Whistleblowing or reporting of illegal activity
- Threats of violence or hostile workplace situations
- Wrongful dismissal or retaliation claims
- Any situation that could expose the organization to legal or reputational risk

For these cases: first send_sensitive_alert_email, then also escalate_to_hr_human.

IMPORTANT: 
- Never guess or invent policy details. When in doubt, escalate
- Never attempt to answer sensitive legal or compliance questions yourself
- Always acknowledge the employee empathetically and assure them HR will follow up

Current user: {current_user}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("placeholder", "{chat_history}"),          # ✅ correct modern syntax
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),      # ✅ correct modern syntax
])

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True,
    return_intermediate_steps=True     # useful for audit logging
)

# # ── TEST INVOCATION ────────────────────────────────────────────────────────────
# response = agent_executor.invoke({
#     "input": "Is MS-Square Limited LLC a listed entity?",
#     "current_user": "john.doe@mssquare.com",
#     "chat_history": []
# })

# response = agent_executor.invoke({
#     "input": "Is MS-Square Limited LLC a fraud entity?",
#     "current_user": "john-doe@ms-square.com",
#     "chat_history": []
# })


# print(response["output"])

# ── CRITICAL: Tell MLflow what to log ─────────────────────────────────────────
mlflow.models.set_model(agent_executor)